In [ ]:

import base64, json, os, pathlib, subprocess, sys, time, torch
T0 = time.monotonic(); CAP = 5400.0; CUT = 4300.0
w = pathlib.Path("/kaggle/working"); p = w / "payload"; p.mkdir(exist_ok=True)
BLOB = {'heat_rad_runner.py': 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJFWFAtMDAxMSAocHJlLXJlZ2lzdGVyZWQgMjAyNi0xMC0wMyk6IFJBRCBhcm0gb24gdGhlIEVYUC0wMDA4IEhlYXQgUkJGIGNvbnRyb2wuCgpJZGVudGljYWwgdG8gdGhlIGNvbnRyb2wgKGhlYXRfc3RlcG1hdGNoZWRfcnVubmVyLnB5KTogc2FtZSBzZWVkLXdpc2UgaW5pdGlhbCB3ZWlnaHRzLCBzYW1lIHNhdmVkIFNvYm9sIHBvb2wgYXMgdGhlIHN0YXJ0aW5nIHBvaW50LCBmdWxsLWJhdGNoIDQsMDk2CmludGVyaW9yIHBvaW50cywgZmxvYXQ2NCwgQWRhbSBsciAxZS0zIGZvciAyMCwwMDAgdXBkYXRlcywgcmVzaWR1YWwgKHVfdCAtIDAuMSB1X3h4KS8xLjEsIGhhcmQgdHJhY2Ugc2luKHBpIHgpK3QgeCAoMS14KSByYXcuCk9ubHkgdGhlIGludGVyaW9yIHBvaW50cyBjaGFuZ2U6IGV2ZXJ5IFJFRlJFU0ggc3RlcHMgdGhlIHxyZXNpZHVhbHwgaXMgZXZhbHVhdGVkIG9uIGEgZml4ZWQgMTYsMzg0LXBvaW50IFNvYm9sIGNhbmRpZGF0ZSBzZXQgYW5kIDQsMDk2IHBvaW50cyBhcmUKcmUtc2FtcGxlZCB3aXRoIHNyYy9tZWNoYW5pc21zL3JhZC5weTo6c2FtcGxlX3JhZCAocG93ZXIgMiwgdW5pZm9ybV9mcmFjdGlvbiAwLjUsIG5vIGltcG9ydGFuY2UgY29ycmVjdGlvbikuIE5vIHNvbHV0aW9uIHZhbHVlcywgZXZhbHVhdGlvbiBncmlkcyBvcgp2ZXJpZmllciBjb2RlIGFyZSB1c2VkIGJ5IHRoZSBtZWNoYW5pc20uIEFmdGVyIEFkYW06IHNjb3JlIChlbmQtb2YtQWRhbSwgdGhlIHByaW1hcnkgbWV0cmljKSwgdGhlbiBmdWxsIEwtQkZHUyAoc3RyZXNzIHRlc3QpIG9uIHRoZSBsYXN0IFJBRCBwb2ludHMuClVzYWdlOiBweXRob24gaGVhdF9yYWRfcnVubmVyLnB5IFNFRUQgLS1kaXIgcGF5bG9hZCAtLW91dCByZXN1bHRzIFstLWFkYW0tc3RlcHMgMjAwMDBdIFstLWRldmljZSBjdWRhOjBdIFstLWxiZmdzLXNlY29uZHMgNzAwXQoiIiIKaW1wb3J0IGFyZ3BhcnNlLCBqc29uLCBvcywgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCm9zLmVudmlyb24uc2V0ZGVmYXVsdCgiQ1VCTEFTX1dPUktTUEFDRV9DT05GSUciLCAiOjQwOTY6OCIpCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKZnJvbSBzY2lweS5zdGF0cyBpbXBvcnQgcW1jCgphcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKYXAuYWRkX2FyZ3VtZW50KCJzZWVkIiwgdHlwZT1pbnQpCmFwLmFkZF9hcmd1bWVudCgiLS1kaXIiLCBkZWZhdWx0PSJwYXlsb2FkIikKYXAuYWRkX2FyZ3VtZW50KCItLW91dCIsIGRlZmF1bHQ9InJlc3VsdHMiKQphcC5hZGRfYXJndW1lbnQoIi0tYWRhbS1zdGVwcyIsIHR5cGU9aW50LCBkZWZhdWx0PTIwMDAwKQphcC5hZGRfYXJndW1lbnQoIi0tcmVmcmVzaCIsIHR5cGU9aW50LCBkZWZhdWx0PTEwMDApCmFwLmFkZF9hcmd1bWVudCgiLS1jYW5kaWRhdGVzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTYzODQpCmFwLmFkZF9hcmd1bWVudCgiLS1wb3dlciIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9Mi4wKQphcC5hZGRfYXJndW1lbnQoIi0tdW5pZm9ybSIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC41KQphcC5hZGRfYXJndW1lbnQoIi0tbGJmZ3Mtc2Vjb25kcyIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9NzAwLjApCmFwLmFkZF9hcmd1bWVudCgiLS1kZXZpY2UiLCBkZWZhdWx0PSJjdWRhOjAiKQphID0gYXAucGFyc2VfYXJncygpCgppbXBvcnQgc3lzCnN5cy5wYXRoLmluc2VydCgwLCBhLmRpcikKZnJvbSBhZl9zY2ltbF9tb2RlbCBpbXBvcnQgYnVpbGRfbW9kZWwgICMgbm9xYTogRTQwMgpmcm9tIHJhZCBpbXBvcnQgc2FtcGxlX3JhZCAgIyBub3FhOiBFNDAyICAoY29weSBvZiBzcmMvbWVjaGFuaXNtcy9yYWQucHkpCgpBTFBIQSwgUlMgPSAwLjEsIDEuMQpTID0gYS5zZWVkCmRldiA9IHRvcmNoLmRldmljZShhLmRldmljZSkKdG9yY2gubWFudWFsX3NlZWQoUyk7IG5wLnJhbmRvbS5zZWVkKFMpCnRvcmNoLnNldF9kZWZhdWx0X2R0eXBlKHRvcmNoLmZsb2F0NjQpCm91dCA9IFBhdGgoYS5vdXQpIC8gZiJzZWVkLXtTfSIKb3V0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKeiA9IG5wLmxvYWQoUGF0aChhLmRpcikgLyBmImhlYXQtc2VlZC17U30ubnB6IikKcG9vbDAgPSBucC5hcnJheSh6WyJpbnRlcmlvciJdLCBkdHlwZT1ucC5mbG9hdDY0KQptb2RlbCA9IGJ1aWxkX21vZGVsKCJoZWF0IiwgeyJpZCI6ICJyYmZfZml4ZWQifSkudG8oZGV2aWNlPWRldiwgZHR5cGU9dG9yY2guZmxvYXQ2NCkKcGFyYW1zID0gc3VtKHAubnVtZWwoKSBmb3IgcCBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkgaWYgcC5yZXF1aXJlc19ncmFkKQpjYW5kX25wID0gcW1jLlNvYm9sKDIsIHNjcmFtYmxlPVRydWUsIHNlZWQ9MTAgKiBTICsgNykucmFuZG9tX2Jhc2UyKGludChucC5sb2cyKGEuY2FuZGlkYXRlcykpKQpjYW5kID0gdG9yY2gudGVuc29yKGNhbmRfbnAsIGR0eXBlPXRvcmNoLmZsb2F0NjQsIGRldmljZT1kZXYpCnJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZygxMCAqIFMgKyA4KQoKCmRlZiByZXNpZHVhbChwdHMpOgogICAgcHRzID0gcHRzLmNsb25lKCkucmVxdWlyZXNfZ3JhZF8oVHJ1ZSkKICAgIG8gPSBtb2RlbChwdHMpCiAgICBnID0gdG9yY2guYXV0b2dyYWQuZ3JhZChvLCBwdHMsIHRvcmNoLm9uZXNfbGlrZShvKSwgY3JlYXRlX2dyYXBoPVRydWUsIHJldGFpbl9ncmFwaD1UcnVlKVswXQogICAgdXh4ID0gdG9yY2guYXV0b2dyYWQuZ3JhZChnWzosIDA6MV0sIHB0cywgdG9yY2gub25lc19saWtlKGdbOiwgMDoxXSksIGNyZWF0ZV9ncmFwaD1UcnVlLCByZXRhaW5fZ3JhcGg9VHJ1ZSlbMF1bOiwgMDoxXQogICAgcmV0dXJuIChnWzosIDE6Ml0gLSBBTFBIQSAqIHV4eCkgLyBSUwoKCmRlZiBsb3NzX29uKHBvaW50cyk6CiAgICByZXR1cm4gcmVzaWR1YWwocG9pbnRzKS5zcXVhcmUoKS5tZWFuKCkKCgpkZWYgdHJhcChuKToKICAgIHcgPSB0b3JjaC5vbmVzKG4sIGR0eXBlPXRvcmNoLmZsb2F0NjQsIGRldmljZT1kZXYpOyB3WzBdID0gd1stMV0gPSAwLjUKICAgIHJldHVybiB3IC8gKG4gLSAxKQoKCmRlZiBzY29yZSgpOgogICAgbngsIG50ID0gNTEyLCAyNTYKICAgIHggPSB0b3JjaC5saW5zcGFjZSgwLCAxLCBueCwgZGV2aWNlPWRldiwgZHR5cGU9dG9yY2guZmxvYXQ2NCk7IHQgPSB0b3JjaC5saW5zcGFjZSgwLCAxLCBudCwgZGV2aWNlPWRldiwgZHR5cGU9dG9yY2guZmxvYXQ2NCkKICAgIHh4LCB0dCA9IHRvcmNoLm1lc2hncmlkKHgsIHQsIGluZGV4aW5nPSJpaiIpCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICBwcmVkID0gbW9kZWwodG9yY2guc3RhY2soKHh4LnJlc2hhcGUoLTEpLCB0dC5yZXNoYXBlKC0xKSksIDEpKS5yZXNoYXBlKG54LCBudCkKICAgIGV4ID0gdG9yY2guc2luKHRvcmNoLnBpICogeHgpICogdG9yY2guZXhwKC1BTFBIQSAqIHRvcmNoLnBpICoqIDIgKiB0dCkKICAgIHcgPSB0cmFwKG54KVs6LCBOb25lXSAqIHRyYXAobnQpW05vbmUsIDpdCiAgICByZXR1cm4gZmxvYXQodG9yY2guc3FydCh0b3JjaC5zdW0odyAqIChwcmVkIC0gZXgpICoqIDIpIC8gdG9yY2guc3VtKHcgKiBleCAqKiAyKSkpCgoKZGVmIHN5bmMoKToKICAgIGlmIGRldi50eXBlID09ICJjdWRhIjoKICAgICAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCkKCgpwb2ludHMgPSB0b3JjaC50ZW5zb3IocG9vbDAsIGR0eXBlPXRvcmNoLmZsb2F0NjQsIGRldmljZT1kZXYpCm9wdCA9IHRvcmNoLm9wdGltLkFkYW0obW9kZWwucGFyYW1ldGVycygpLCBscj0xZS0zKQpzeW5jKCk7IHQwID0gdGltZS5wZXJmX2NvdW50ZXIoKTsgcmVmcmVzaF9zID0gMC4wOyByZWZyZXNoZXMgPSBbXTsgaGlzdCA9IFtdCmZvciBzdGVwIGluIHJhbmdlKDEsIGEuYWRhbV9zdGVwcyArIDEpOgogICAgaWYgYS5yZWZyZXNoIGFuZCBzdGVwID4gMSBhbmQgKHN0ZXAgLSAxKSAlIGEucmVmcmVzaCA9PSAwOgogICAgICAgIHN5bmMoKTsgdHIgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgcGFydHMgPSBbXQogICAgICAgIGZvciBpIGluIHJhbmdlKDAsIGxlbihjYW5kKSwgNDA5Nik6CiAgICAgICAgICAgIHBhcnRzLmFwcGVuZChyZXNpZHVhbChjYW5kW2k6aSArIDQwOTZdKS5kZXRhY2goKS5hYnMoKS5jcHUoKSkKICAgICAgICByID0gdG9yY2guY2F0KHBhcnRzKS5udW1weSgpLnJlc2hhcGUoLTEpCiAgICAgICAgcyA9IHNhbXBsZV9yYWQoY2FuZF9ucCwgciwgbGVuKHBvb2wwKSwgcm5nLCBwb3dlcj1hLnBvd2VyLCB1bmlmb3JtX2ZyYWN0aW9uPWEudW5pZm9ybSkKICAgICAgICBwb2ludHMgPSB0b3JjaC50ZW5zb3Ioc1sicG9pbnRzIl0sIGR0eXBlPXRvcmNoLmZsb2F0NjQsIGRldmljZT1kZXYpCiAgICAgICAgc3luYygpOyBkdCA9IHRpbWUucGVyZl9jb3VudGVyKCkgLSB0cjsgcmVmcmVzaF9zICs9IGR0CiAgICAgICAgcmVmcmVzaGVzLmFwcGVuZCh7InN0ZXAiOiBzdGVwIC0gMSwgInVuaXF1ZSI6IHNbInVuaXF1ZV9jb3VudCJdLCAiZWZmZWN0aXZlX2NhbmRpZGF0ZXMiOiBzWyJlZmZlY3RpdmVfY2FuZGlkYXRlcyJdLCAic2Vjb25kcyI6IGR0fSkKICAgIG9wdC56ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKICAgIGwgPSBsb3NzX29uKHBvaW50cykKICAgIGwuYmFja3dhcmQoKQogICAgb3B0LnN0ZXAoKQogICAgaWYgc3RlcCAlIDUwMCA9PSAwOgogICAgICAgIGhpc3QuYXBwZW5kKHsic3RlcCI6IHN0ZXAsICJsb3NzIjogZmxvYXQobC5kZXRhY2goKSl9KQpzeW5jKCk7IGFkYW1fcyA9IHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MAphZGFtX3Njb3JlID0gc2NvcmUoKQp0b3JjaC5zYXZlKHsic3RhdGVfZGljdCI6IG1vZGVsLnN0YXRlX2RpY3QoKSwgInN0YWdlIjogImFkYW1fZmluYWxfcmFkIn0sIG91dCAvICJjaGVja3BvaW50LWFkYW0tZmluYWwucHQiKQpwcmludChqc29uLmR1bXBzKHsic2VlZCI6IFMsICJwYXJhbXMiOiBwYXJhbXMsICJhZGFtX3JlbF9sMiI6IGFkYW1fc2NvcmUsICJhZGFtX3MiOiBhZGFtX3MsICJyZWZyZXNoX3MiOiByZWZyZXNoX3N9KSwgZmx1c2g9VHJ1ZSkKCkNIVU5LLCBNQVhFVkFMLCBNQVhJVEVSID0gMjUsIDEyNSwgMTUwMDAKbGIgPSB0b3JjaC5vcHRpbS5MQkZHUyhtb2RlbC5wYXJhbWV0ZXJzKCksIGxyPTEuMCwgbWF4X2l0ZXI9Q0hVTkssIG1heF9ldmFsPU1BWEVWQUwsIHRvbGVyYW5jZV9ncmFkPTFlLTgsIHRvbGVyYW5jZV9jaGFuZ2U9MC4wLCBoaXN0b3J5X3NpemU9MTAwLCBsaW5lX3NlYXJjaF9mbj0ic3Ryb25nX3dvbGZlIikKCgpkZWYgY2xvc3VyZSgpOgogICAgbGIuemVyb19ncmFkKHNldF90b19ub25lPVRydWUpCiAgICB2ID0gbG9zc19vbihwb2ludHMpCiAgICB2LmJhY2t3YXJkKCkKICAgIHJldHVybiB2CgoKbGFzdCA9IGZsb2F0KGxvc3Nfb24ocG9pbnRzKS5kZXRhY2goKSk7IGl0ID0gMDsgcGxhdGVhdSA9IDA7IHJlYXNvbiA9ICJtYXhfaXRlcmF0aW9ucyI7IHQxID0gdGltZS5wZXJmX2NvdW50ZXIoKQp3aGlsZSBpdCA8IE1BWElURVI6CiAgICBpZiB0aW1lLnBlcmZfY291bnRlcigpIC0gdDEgPj0gYS5sYmZnc19zZWNvbmRzOgogICAgICAgIHJlYXNvbiA9ICJ0aW1lX2NhcCI7IGJyZWFrCiAgICBiID0gaW50KGxiLnN0YXRlW2xiLl9wYXJhbXNbMF1dLmdldCgibl9pdGVyIiwgMCkpCiAgICBsYi5zdGVwKGNsb3N1cmUpCiAgICBkID0gaW50KGxiLnN0YXRlW2xiLl9wYXJhbXNbMF1dLmdldCgibl9pdGVyIiwgYikpIC0gYgogICAgaXQgKz0gZAogICAgY3VyID0gZmxvYXQobG9zc19vbihwb2ludHMpLmRldGFjaCgpKQogICAgaWYgbm90IG5wLmlzZmluaXRlKGN1cik6CiAgICAgICAgcmVhc29uID0gIm5vbmZpbml0ZSI7IGJyZWFrCiAgICBpZiBkID09IDA6CiAgICAgICAgcmVhc29uID0gIm5vX3Byb2dyZXNzX2NodW5rIjsgYnJlYWsKICAgIHBsYXRlYXUgPSBwbGF0ZWF1ICsgMSBpZiAobGFzdCAtIGN1cikgLyBtYXgoYWJzKGxhc3QpLCAxZS0zMDApIDwgMWUtMTAgZWxzZSAwCiAgICBsYXN0ID0gY3VyCiAgICBpZiBwbGF0ZWF1ID49IDU6CiAgICAgICAgcmVhc29uID0gImxvc3NfcGxhdGVhdSI7IGJyZWFrCnN5bmMoKQpmaW5hbCA9IHsic2VlZCI6IFMsICJleHBlcmltZW50IjogIkVYUC0wMDExIiwgIm1lY2hhbmlzbSI6ICJyYWQgcG93ZXI9JWcgdW5pZm9ybT0lZyByZWZyZXNoPSVkIGNhbmRpZGF0ZXM9JWQiICUgKGEucG93ZXIsIGEudW5pZm9ybSwgYS5yZWZyZXNoLCBhLmNhbmRpZGF0ZXMpLAogICAgICAgICAicGFyYW1zIjogcGFyYW1zLCAiYWRhbV9zdGVwcyI6IGEuYWRhbV9zdGVwcywgImFkYW1fc2Vjb25kcyI6IGFkYW1fcywgInJlZnJlc2hfc2Vjb25kcyI6IHJlZnJlc2hfcywgImVuZF9vZl9hZGFtX3JlbF9sMl81MTJ4MjU2IjogYWRhbV9zY29yZSwKICAgICAgICAgImxiZmdzX2l0ZXJzIjogaXQsICJsYmZnc19zdG9wX3JlYXNvbiI6IHJlYXNvbiwgImxiZmdzX3NlY29uZHMiOiB0aW1lLnBlcmZfY291bnRlcigpIC0gdDEsICJhZnRlcl9sYmZnc19yZWxfbDJfNTEyeDI1NiI6IHNjb3JlKCksCiAgICAgICAgICJyZWZyZXNoZXMiOiByZWZyZXNoZXNbOjNdICsgcmVmcmVzaGVzWy0yOl0sICJuX3JlZnJlc2hlcyI6IGxlbihyZWZyZXNoZXMpLCAibG9zc19oaXN0b3J5IjogaGlzdCwgInBvb2xfc2hhMjU2Ijogc3RyKHpbInBvb2xfc2hhMjU2Il0pLAogICAgICAgICAiZGV2aWNlIjogdG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoZGV2KSBpZiBkZXYudHlwZSA9PSAiY3VkYSIgZWxzZSAiY3B1IiwgInNjaWVudGlmaWNfYWNjZXB0ZWQiOiBGYWxzZSwgInJlZmVyZW5jZV9hY2NlcHRlZCI6IEZhbHNlfQp0b3JjaC5zYXZlKHsic3RhdGVfZGljdCI6IG1vZGVsLnN0YXRlX2RpY3QoKSwgInN0YWdlIjogImxiZmdzX2ZpbmFsX3JhZCIsICJpdGVycyI6IGl0fSwgb3V0IC8gImNoZWNrcG9pbnQtbGJmZ3MtZmluYWwucHQiKQpqc29uLmR1bXAoZmluYWwsIG9wZW4ob3V0IC8gInJlc3VsdC5qc29uIiwgInciKSwgaW5kZW50PTIpCnByaW50KCJGSU5BTCIsIGpzb24uZHVtcHMoe2s6IHYgZm9yIGssIHYgaW4gZmluYWwuaXRlbXMoKSBpZiBrIG5vdCBpbiAoInJlZnJlc2hlcyIsICJsb3NzX2hpc3RvcnkiKX0pLCBmbHVzaD1UcnVlKQo=', 'af_sciml_model.py': 'IiIiUGFyYW1ldGVyLW1hdGNoZWQgbmV1cmFsIGFuZCBLb2xtb2dvcm92LUFybm9sZCBtb2RlbHMgZm9yIFNjaU1MIHNjcmVlbmluZy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoIGltcG9ydCBubgpmcm9tIHRvcmNoLm5uIGltcG9ydCBmdW5jdGlvbmFsIGFzIEYKCgpjbGFzcyBSQkZMYXllcihubi5Nb2R1bGUpOgogICAgIiIiRml4ZWQgR2F1c3NpYW4gcmFkaWFsIGJhc2VzIHBsdXMgYSB0cmFpbmFibGUgU2lMVSBlZGdlIGZ1bmN0aW9uLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbnB1dHM6IGludCwgb3V0cHV0czogaW50LCBiYXNpc19jb3VudDogaW50ID0gMTEpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYucmVnaXN0ZXJfYnVmZmVyKCJjZW50ZXJzIiwgdG9yY2gubGluc3BhY2UoLTEuMDUsIDEuMDUsIGJhc2lzX2NvdW50KSkKICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigid2lkdGgiLCB0b3JjaC50ZW5zb3IoMC4yMSkpCiAgICAgICAgc2VsZi5jb2VmZmljaWVudHMgPSBubi5QYXJhbWV0ZXIodG9yY2guZW1wdHkob3V0cHV0cywgaW5wdXRzLCBiYXNpc19jb3VudCkpCiAgICAgICAgc2VsZi5iYXNlID0gbm4uUGFyYW1ldGVyKHRvcmNoLmVtcHR5KG91dHB1dHMsIGlucHV0cykpCiAgICAgICAgbm4uaW5pdC5ub3JtYWxfKHNlbGYuY29lZmZpY2llbnRzLCBzdGQ9MC4xIC8gKGlucHV0cyoqMC41KSkKICAgICAgICBubi5pbml0Lm5vcm1hbF8oc2VsZi5iYXNlLCBzdGQ9MC4xIC8gKGlucHV0cyoqMC41KSkKCiAgICBkZWYgYmFzaXMoc2VsZiwgeDogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgcmV0dXJuIHRvcmNoLmV4cCgtKCh4LnVuc3F1ZWV6ZSgtMSkgLSBzZWxmLmNlbnRlcnMpIC8gc2VsZi53aWR0aCkgKiogMikKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICByZXR1cm4gdG9yY2guZWluc3VtKCJiaWcsb2lnLT5ibyIsIHNlbGYuYmFzaXMoeCksIHNlbGYuY29lZmZpY2llbnRzKSArIEYubGluZWFyKAogICAgICAgICAgICBGLnNpbHUoeCksIHNlbGYuYmFzZQogICAgICAgICkKCgpjbGFzcyBCU3BsaW5lTGF5ZXIobm4uTW9kdWxlKToKICAgICIiIkN1YmljIG9wZW4tdW5pZm9ybSBCLXNwbGluZSBlZGdlIGZ1bmN0aW9ucyB3aXRoIGZpeGVkIGtub3RzLiIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIGlucHV0czogaW50LAogICAgICAgIG91dHB1dHM6IGludCwKICAgICAgICBiYXNpc19jb3VudDogaW50ID0gMTEsCiAgICAgICAgZGVncmVlOiBpbnQgPSAzLAogICAgICAgIGxvdzogZmxvYXQgPSAtMS4wNSwKICAgICAgICBoaWdoOiBmbG9hdCA9IDEuMDUsCiAgICApOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIGlmIGJhc2lzX2NvdW50IDw9IGRlZ3JlZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiYmFzaXNfY291bnQgbXVzdCBleGNlZWQgc3BsaW5lIGRlZ3JlZSIpCiAgICAgICAgaWYgbm90IGxvdyA8IGhpZ2g6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInNwbGluZSBpbnRlcnZhbCBtdXN0IGhhdmUgcG9zaXRpdmUgd2lkdGgiKQogICAgICAgIGludGVyaW9yX2NvdW50ID0gYmFzaXNfY291bnQgLSBkZWdyZWUgLSAxCiAgICAgICAgaW50ZXJpb3IgPSAoCiAgICAgICAgICAgIHRvcmNoLmxpbnNwYWNlKGxvdywgaGlnaCwgaW50ZXJpb3JfY291bnQgKyAyKVsxOi0xXQogICAgICAgICAgICBpZiBpbnRlcmlvcl9jb3VudAogICAgICAgICAgICBlbHNlIHRvcmNoLmVtcHR5KDApCiAgICAgICAgKQogICAgICAgIGtub3RzID0gdG9yY2guY2F0KAogICAgICAgICAgICAoCiAgICAgICAgICAgICAgICB0b3JjaC5mdWxsKChkZWdyZWUgKyAxLCksIGxvdyksCiAgICAgICAgICAgICAgICBpbnRlcmlvciwKICAgICAgICAgICAgICAgIHRvcmNoLmZ1bGwoKGRlZ3JlZSArIDEsKSwgaGlnaCksCiAgICAgICAgICAgICkKICAgICAgICApCiAgICAgICAgc2VsZi5yZWdpc3Rlcl9idWZmZXIoImtub3RzIiwga25vdHMpCiAgICAgICAgc2VsZi5kZWdyZWUgPSBkZWdyZWUKICAgICAgICBzZWxmLmNvZWZmaWNpZW50cyA9IG5uLlBhcmFtZXRlcih0b3JjaC5lbXB0eShvdXRwdXRzLCBpbnB1dHMsIGJhc2lzX2NvdW50KSkKICAgICAgICBzZWxmLmJhc2UgPSBubi5QYXJhbWV0ZXIodG9yY2guZW1wdHkob3V0cHV0cywgaW5wdXRzKSkKICAgICAgICBubi5pbml0Lm5vcm1hbF8oc2VsZi5jb2VmZmljaWVudHMsIHN0ZD0wLjEgLyAoaW5wdXRzKiowLjUpKQogICAgICAgIG5uLmluaXQubm9ybWFsXyhzZWxmLmJhc2UsIHN0ZD0wLjEgLyAoaW5wdXRzKiowLjUpKQoKICAgIGRlZiBiYXNpcyhzZWxmLCB4OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICAiIiJFdmFsdWF0ZSB0aGUgQ294LWRlIEJvb3IgYmFzaXMsIHJldGFpbmluZyBpbnB1dCBhdXRvZ3JhZCBkZXJpdmF0aXZlcy4iIiIKICAgICAgICBjb29yZGluYXRlID0geC51bnNxdWVlemUoLTEpCiAgICAgICAgYmFzaXMgPSAoKGNvb3JkaW5hdGUgPj0gc2VsZi5rbm90c1s6LTFdKSAmIChjb29yZGluYXRlIDwgc2VsZi5rbm90c1sxOl0pKS50byh4LmR0eXBlKQogICAgICAgICMgSW5wdXRzIGluIHRoaXMgZXhwZXJpbWVudCBsaWUgc3RyaWN0bHkgaW5zaWRlIHRoZSBwYWRkZWQga25vdCBpbnRlcnZhbC4KICAgICAgICAjIEtlZXBpbmcgdGhlIGVuZHBvaW50IGNvbnZlbnRpb24gbWFrZXMgdGhlIGxheWVyIHdlbGwtZGVmaW5lZCBpZiB0aGUKICAgICAgICAjIGludGVydmFsIGlzIGxhdGVyIHVzZWQgZGlyZWN0bHkgYXMgYSBtb2RlbCBkb21haW4uCiAgICAgICAgYXRfcmlnaHRfZW5kcG9pbnQgPSBjb29yZGluYXRlLnNxdWVlemUoLTEpID09IHNlbGYua25vdHNbLTFdCiAgICAgICAgaWYgYXRfcmlnaHRfZW5kcG9pbnQuYW55KCk6CiAgICAgICAgICAgIGVuZHBvaW50ID0gdG9yY2guemVyb3NfbGlrZShiYXNpcykKICAgICAgICAgICAgZW5kcG9pbnRbLi4uLCAtMV0gPSBhdF9yaWdodF9lbmRwb2ludC50byh4LmR0eXBlKQogICAgICAgICAgICBiYXNpcyA9IHRvcmNoLndoZXJlKGF0X3JpZ2h0X2VuZHBvaW50LnVuc3F1ZWV6ZSgtMSksIGVuZHBvaW50LCBiYXNpcykKCiAgICAgICAgZm9yIG9yZGVyIGluIHJhbmdlKDEsIHNlbGYuZGVncmVlICsgMSk6CiAgICAgICAgICAgIGxlZnRfZGVub21pbmF0b3IgPSBzZWxmLmtub3RzW29yZGVyOi0xXSAtIHNlbGYua25vdHNbOiAtKG9yZGVyICsgMSldCiAgICAgICAgICAgIHJpZ2h0X2Rlbm9taW5hdG9yID0gc2VsZi5rbm90c1tvcmRlciArIDEgOl0gLSBzZWxmLmtub3RzWzE6LW9yZGVyXQogICAgICAgICAgICBsZWZ0X3dlaWdodCA9IChjb29yZGluYXRlIC0gc2VsZi5rbm90c1s6IC0ob3JkZXIgKyAxKV0pIC8gbGVmdF9kZW5vbWluYXRvci5jbGFtcF9taW4oCiAgICAgICAgICAgICAgICB0b3JjaC5maW5mbyh4LmR0eXBlKS5lcHMKICAgICAgICAgICAgKQogICAgICAgICAgICByaWdodF93ZWlnaHQgPSAoc2VsZi5rbm90c1tvcmRlciArIDEgOl0gLSBjb29yZGluYXRlKSAvIHJpZ2h0X2Rlbm9taW5hdG9yLmNsYW1wX21pbigKICAgICAgICAgICAgICAgIHRvcmNoLmZpbmZvKHguZHR5cGUpLmVwcwogICAgICAgICAgICApCiAgICAgICAgICAgIGxlZnRfd2VpZ2h0ID0gdG9yY2gud2hlcmUobGVmdF9kZW5vbWluYXRvciA+IDAsIGxlZnRfd2VpZ2h0LCAwLjApCiAgICAgICAgICAgIHJpZ2h0X3dlaWdodCA9IHRvcmNoLndoZXJlKHJpZ2h0X2Rlbm9taW5hdG9yID4gMCwgcmlnaHRfd2VpZ2h0LCAwLjApCiAgICAgICAgICAgIGJhc2lzID0gbGVmdF93ZWlnaHQgKiBiYXNpc1suLi4sIDotMV0gKyByaWdodF93ZWlnaHQgKiBiYXNpc1suLi4sIDE6XQogICAgICAgIHJldHVybiBiYXNpcwoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHg6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIHJldHVybiB0b3JjaC5laW5zdW0oImJpZyxvaWctPmJvIiwgc2VsZi5iYXNpcyh4KSwgc2VsZi5jb2VmZmljaWVudHMpICsgRi5saW5lYXIoCiAgICAgICAgICAgIEYuc2lsdSh4KSwgc2VsZi5iYXNlCiAgICAgICAgKQoKCmNsYXNzIF9LQU5CYXNlKG5uLk1vZHVsZSk6CiAgICBtb2RlbF9mYW1pbHk6IHN0cgoKICAgIGRlZiBub3JtYWxpemVkKHNlbGYsIHg6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIGlmIGhhc2F0dHIoeCwgInRlbnNvciIpOgogICAgICAgICAgICB4ID0geC50ZW5zb3IKICAgICAgICB4X3NwYWNlID0gMi4wICogeFs6LCAwOjFdIC0gMS4wIGlmIHNlbGYucHJvYmxlbSA9PSAiaGVhdCIgZWxzZSB4WzosIDA6MV0KICAgICAgICByZXR1cm4gdG9yY2guY2F0KCh4X3NwYWNlLCAyLjAgKiB4WzosIDE6Ml0gLSAxLjApLCBkaW09MSkKCiAgICBkZWYgY29uc3RyYWluKHNlbGYsIHg6IHRvcmNoLlRlbnNvciwgcmF3OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICBpZiBzZWxmLnByb2JsZW0gPT0gImhlYXQiOgogICAgICAgICAgICByZXR1cm4gdG9yY2guc2luKHRvcmNoLnBpICogeFs6LCAwOjFdKSArIHhbOiwgMToyXSAqIHhbOiwgMDoxXSAqICgxLjAgLSB4WzosIDA6MV0pICogcmF3CiAgICAgICAgaWYgc2VsZi5wcm9ibGVtID09ICJhbGxlbl9jYWhuIjoKICAgICAgICAgICAgc3BhY2UgPSB4WzosIDA6MV0KICAgICAgICAgICAgdGltZSA9IHhbOiwgMToyXQogICAgICAgICAgICBpbml0aWFsID0gc3BhY2Uuc3F1YXJlKCkgKiB0b3JjaC5jb3ModG9yY2gucGkgKiBzcGFjZSkKICAgICAgICAgICAgcmV0dXJuIGluaXRpYWwgKyB0aW1lICogKDEuMCAtIHNwYWNlLnNxdWFyZSgpKSAqIHJhdwogICAgICAgIHJldHVybiByYXcKCiAgICBkZWYgZm9yd2FyZF93aXRoX3RyYWNlKAogICAgICAgIHNlbGYsIHg6IHRvcmNoLlRlbnNvcgogICAgKSAtPiB0dXBsZVt0b3JjaC5UZW5zb3IsIGxpc3RbZGljdFtzdHIsIHRvcmNoLlRlbnNvcl1dXToKICAgICAgICBpZiBoYXNhdHRyKHgsICJ0ZW5zb3IiKToKICAgICAgICAgICAgeCA9IHgudGVuc29yCiAgICAgICAgaCA9IHNlbGYubm9ybWFsaXplZCh4KQogICAgICAgIHRyYWNlczogbGlzdFtkaWN0W3N0ciwgdG9yY2guVGVuc29yXV0gPSBbXQogICAgICAgIGZvciBpbmRleCwgbGF5ZXIgaW4gZW51bWVyYXRlKHNlbGYubGF5ZXJzKToKICAgICAgICAgICAgdHJhY2U6IGRpY3Rbc3RyLCB0b3JjaC5UZW5zb3JdID0ge30KICAgICAgICAgICAgaWYgaGFzYXR0cihsYXllciwgImJhc2lzIik6CiAgICAgICAgICAgICAgICB0cmFjZVsiYmFzaXMiXSA9IGxheWVyLmJhc2lzKGgpCiAgICAgICAgICAgIHByZWFjdGl2YXRpb24gPSBsYXllcihoKQogICAgICAgICAgICB0cmFjZVsicHJlYWN0aXZhdGlvbiJdID0gcHJlYWN0aXZhdGlvbgogICAgICAgICAgICBoID0gdG9yY2gudGFuaChwcmVhY3RpdmF0aW9uKSBpZiBpbmRleCA8IGxlbihzZWxmLmxheWVycykgLSAxIGVsc2UgcHJlYWN0aXZhdGlvbgogICAgICAgICAgICB0cmFjZVsiYWN0aXZhdGlvbiJdID0gaAogICAgICAgICAgICB0cmFjZXMuYXBwZW5kKHRyYWNlKQogICAgICAgIHJldHVybiBzZWxmLmNvbnN0cmFpbih4LCBoKSwgdHJhY2VzCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeDogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgb3V0cHV0LCBfID0gc2VsZi5mb3J3YXJkX3dpdGhfdHJhY2UoeCkKICAgICAgICByZXR1cm4gb3V0cHV0CgoKY2xhc3MgUkJGUElLQU4oX0tBTkJhc2UpOgogICAgIiIiRml4ZWQtY2VudGVyIFJCRiBLQU4sIHJldGFpbmVkIGFzIHRoZSBleGlzdGluZyBtb2RlbCBjb250cm9sLiIiIgoKICAgIG1vZGVsX2lkID0gInJiZl9maXhlZCIKICAgIG1vZGVsX2ZhbWlseSA9ICJyYmYiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHByb2JsZW06IHN0ciwgYmFzaXNfY291bnQ6IGludCA9IDExKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBpZiBwcm9ibGVtIG5vdCBpbiAoImhlYXQiLCAiYnVyZ2VycyIsICJhbGxlbl9jYWhuIik6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInVua25vd24gcHJvYmxlbSIpCiAgICAgICAgc2VsZi5wcm9ibGVtID0gcHJvYmxlbQogICAgICAgIHNlbGYubGF5ZXJzID0gbm4uTW9kdWxlTGlzdCgKICAgICAgICAgICAgW1JCRkxheWVyKGEsIGIsIGJhc2lzX2NvdW50KSBmb3IgYSwgYiBpbiB6aXAoWzIsIDE2LCAxNl0sIFsxNiwgMTYsIDFdKV0KICAgICAgICApCgoKY2xhc3MgQlNwbGluZVBJS0FOKF9LQU5CYXNlKToKICAgICIiIkZpeGVkIGN1YmljIEItc3BsaW5lIEtBTiB1c2luZyB0aGUgc2FtZSBlZGdlIGFuZCBsYXllciBkaW1lbnNpb25zLiIiIgoKICAgIG1vZGVsX2lkID0gImJzcGxpbmVfY3ViaWMiCiAgICBtb2RlbF9mYW1pbHkgPSAiYnNwbGluZSIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgcHJvYmxlbTogc3RyLCBiYXNpc19jb3VudDogaW50ID0gMTEsIGRlZ3JlZTogaW50ID0gMyk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgaWYgcHJvYmxlbSBub3QgaW4gKCJoZWF0IiwgImJ1cmdlcnMiLCAiYWxsZW5fY2FobiIpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ1bmtub3duIHByb2JsZW0iKQogICAgICAgIHNlbGYucHJvYmxlbSA9IHByb2JsZW0KICAgICAgICBzZWxmLmxheWVycyA9IG5uLk1vZHVsZUxpc3QoCiAgICAgICAgICAgIFsKICAgICAgICAgICAgICAgIEJTcGxpbmVMYXllcihhLCBiLCBiYXNpc19jb3VudCwgZGVncmVlKQogICAgICAgICAgICAgICAgZm9yIGEsIGIgaW4gemlwKFsyLCAxNiwgMTZdLCBbMTYsIDE2LCAxXSkKICAgICAgICAgICAgXQogICAgICAgICkKCgpjbGFzcyBQSU5OTUxQKF9LQU5CYXNlKToKICAgICIiIlRhbmggUElOTiB3aXRoIGEgaGlkZGVuIHdpZHRoIGNob3NlbiB0byBtYXRjaCB0aGUgUkJGIHBhcmFtZXRlciBjb3VudC4iIiIKCiAgICBtb2RlbF9pZCA9ICJtbHBfdGFuaCIKICAgIG1vZGVsX2ZhbWlseSA9ICJtbHAiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHByb2JsZW06IHN0ciwgd2lkdGg6IGludCA9IDU4KToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBpZiBwcm9ibGVtIG5vdCBpbiAoImhlYXQiLCAiYnVyZ2VycyIsICJhbGxlbl9jYWhuIik6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInVua25vd24gcHJvYmxlbSIpCiAgICAgICAgc2VsZi5wcm9ibGVtID0gcHJvYmxlbQogICAgICAgIHNlbGYud2lkdGggPSB3aWR0aAogICAgICAgIHNlbGYubGF5ZXJzID0gbm4uTW9kdWxlTGlzdCgKICAgICAgICAgICAgW25uLkxpbmVhcigyLCB3aWR0aCksIG5uLkxpbmVhcih3aWR0aCwgd2lkdGgpLCBubi5MaW5lYXIod2lkdGgsIDEpXQogICAgICAgICkKICAgICAgICBmb3IgbGF5ZXIgaW4gc2VsZi5sYXllcnNbOi0xXToKICAgICAgICAgICAgbm4uaW5pdC54YXZpZXJfdW5pZm9ybV8obGF5ZXIud2VpZ2h0LCBnYWluPW5uLmluaXQuY2FsY3VsYXRlX2dhaW4oInRhbmgiKSkKICAgICAgICAgICAgbm4uaW5pdC56ZXJvc18obGF5ZXIuYmlhcykKICAgICAgICBubi5pbml0Lnhhdmllcl91bmlmb3JtXyhzZWxmLmxheWVyc1stMV0ud2VpZ2h0KQogICAgICAgIG5uLmluaXQuemVyb3NfKHNlbGYubGF5ZXJzWy0xXS5iaWFzKQoKICAgIGRlZiBmb3J3YXJkX3dpdGhfdHJhY2UoCiAgICAgICAgc2VsZiwgeDogdG9yY2guVGVuc29yCiAgICApIC0+IHR1cGxlW3RvcmNoLlRlbnNvciwgbGlzdFtkaWN0W3N0ciwgdG9yY2guVGVuc29yXV1dOgogICAgICAgIGlmIGhhc2F0dHIoeCwgInRlbnNvciIpOgogICAgICAgICAgICB4ID0geC50ZW5zb3IKICAgICAgICBoID0gc2VsZi5ub3JtYWxpemVkKHgpCiAgICAgICAgdHJhY2VzOiBsaXN0W2RpY3Rbc3RyLCB0b3JjaC5UZW5zb3JdXSA9IFtdCiAgICAgICAgZm9yIGluZGV4LCBsYXllciBpbiBlbnVtZXJhdGUoc2VsZi5sYXllcnMpOgogICAgICAgICAgICBwcmVhY3RpdmF0aW9uID0gbGF5ZXIoaCkKICAgICAgICAgICAgaCA9IHRvcmNoLnRhbmgocHJlYWN0aXZhdGlvbikgaWYgaW5kZXggPCBsZW4oc2VsZi5sYXllcnMpIC0gMSBlbHNlIHByZWFjdGl2YXRpb24KICAgICAgICAgICAgdHJhY2VzLmFwcGVuZCh7InByZWFjdGl2YXRpb24iOiBwcmVhY3RpdmF0aW9uLCAiYWN0aXZhdGlvbiI6IGh9KQogICAgICAgIHJldHVybiBzZWxmLmNvbnN0cmFpbih4LCBoKSwgdHJhY2VzCgoKZGVmIGJ1aWxkX21vZGVsKHByb2JsZW06IHN0ciwgY29uZmlnOiBkaWN0IHwgTm9uZSA9IE5vbmUpIC0+IG5uLk1vZHVsZToKICAgICIiIkNyZWF0ZSBhbiBleHBsaWNpdGx5IG5hbWVkIG1vZGVsOyBSQkYgcmVtYWlucyB0aGUgYmFja3dhcmRzLWNvbXBhdGlibGUgZGVmYXVsdC4iIiIKICAgIGNvbmZpZyA9IHt9IGlmIGNvbmZpZyBpcyBOb25lIGVsc2UgZGljdChjb25maWcpCiAgICBtb2RlbF9pZCA9IGNvbmZpZy5nZXQoImlkIiwgY29uZmlnLmdldCgibW9kZWxfaWQiLCAicmJmX2ZpeGVkIikpCiAgICBpZiBtb2RlbF9pZCA9PSAicmJmX2ZpeGVkIjoKICAgICAgICByZXR1cm4gUkJGUElLQU4ocHJvYmxlbSwgYmFzaXNfY291bnQ9aW50KGNvbmZpZy5nZXQoImJhc2lzX2NvdW50IiwgMTEpKSkKICAgIGlmIG1vZGVsX2lkID09ICJic3BsaW5lX2N1YmljIjoKICAgICAgICByZXR1cm4gQlNwbGluZVBJS0FOKAogICAgICAgICAgICBwcm9ibGVtLAogICAgICAgICAgICBiYXNpc19jb3VudD1pbnQoY29uZmlnLmdldCgiYmFzaXNfY291bnQiLCAxMSkpLAogICAgICAgICAgICBkZWdyZWU9aW50KGNvbmZpZy5nZXQoImRlZ3JlZSIsIDMpKSwKICAgICAgICApCiAgICBpZiBtb2RlbF9pZCA9PSAibWxwX3RhbmgiOgogICAgICAgIHJldHVybiBQSU5OTUxQKHByb2JsZW0sIHdpZHRoPWludChjb25maWcuZ2V0KCJ3aWR0aCIsIDU4KSkpCiAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5rbm93biBtb2RlbCBpZDoge21vZGVsX2lkIXJ9IikKCgpkZWYgZGVyaXZhdGl2ZXMoeTogdG9yY2guVGVuc29yLCB4OiB0b3JjaC5UZW5zb3IpIC0+IHR1cGxlW3RvcmNoLlRlbnNvciwgLi4uXToKICAgIGdyYWRpZW50ID0gdG9yY2guYXV0b2dyYWQuZ3JhZCgKICAgICAgICB5LCB4LCB0b3JjaC5vbmVzX2xpa2UoeSksIGNyZWF0ZV9ncmFwaD1UcnVlLCByZXRhaW5fZ3JhcGg9VHJ1ZQogICAgKVswXQogICAgdXggPSBncmFkaWVudFs6LCAwOjFdCiAgICB1dCA9IGdyYWRpZW50WzosIDE6Ml0KICAgIHV4eCA9IHRvcmNoLmF1dG9ncmFkLmdyYWQoCiAgICAgICAgdXgsIHgsIHRvcmNoLm9uZXNfbGlrZSh1eCksIGNyZWF0ZV9ncmFwaD1UcnVlLCByZXRhaW5fZ3JhcGg9VHJ1ZQogICAgKVswXVs6LCAwOjFdCiAgICByZXR1cm4gdXgsIHV0LCB1eHgK', 'rad.py': 'IiIiUmVzaWR1YWwtYWRhcHRpdmUgZGlzdHJpYnV0aW9uIHdpdGggZml4ZWQgY291bnQgYW5kIGV4cGxpY2l0IHJhbmRvbSBnZW5lcmF0b3IuCgpObyBzb2x1dGlvbiB2YWx1ZXMsIHZhbGlkYXRpb24gZ3JpZHMgb3IgdmVyaWZpZXIgaW1wb3J0cyBlbnRlciB0aGlzIG1vZHVsZS4KU2FtcGxpbmcgaXMgd2l0aCByZXBsYWNlbWVudDogbXVsdGlwbGljaXR5IGltcGxlbWVudHMgdGhlIHRhcmdldCBkaXNjcmV0ZQpkaXN0cmlidXRpb24uIExvc3MgaXMgaW50ZW50aW9uYWxseSBub3QgaW1wb3J0YW5jZS1jb3JyZWN0ZWQuCiIiIgppbXBvcnQgbnVtcHkgYXMgbnAKCgpkZWYgcmVzaWR1YWxfcHJvYmFiaWxpdGllcyhyZXNpZHVhbCwgcG93ZXI9Mi4sIHVuaWZvcm1fZnJhY3Rpb249LjUpOgogICAgcj1ucC5hc2FycmF5KHJlc2lkdWFsLGR0eXBlPW5wLmZsb2F0NjQpLnJlc2hhcGUoLTEpCiAgICBpZiBub3Qgci5zaXplIG9yIG5vdCBucC5pc2Zpbml0ZShyKS5hbGwoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdyZXNpZHVhbCBtdXN0IGJlIG5vbmVtcHR5IGFuZCBmaW5pdGUnKQogICAgaWYgbm90IG5wLmlzZmluaXRlKHBvd2VyKSBvciBwb3dlcjw9MCBvciBub3QgMDw9dW5pZm9ybV9mcmFjdGlvbjw9MToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdwb3dlciBtdXN0IGJlIHBvc2l0aXZlIGFuZCB1bmlmb3JtX2ZyYWN0aW9uIHdpdGhpbiBbMCwxXScpCiAgICBtYWduaXR1ZGU9bnAuYWJzKHIpCiAgICBzY2FsZT1mbG9hdChtYWduaXR1ZGUubWF4KCkpCiAgICBpZiBzY2FsZT09MDoKICAgICAgICByZXR1cm4gbnAuZnVsbChyLnNpemUsMS9yLnNpemUpCiAgICB3ZWlnaHRzPShtYWduaXR1ZGUvc2NhbGUpKipwb3dlcgogICAgcD11bmlmb3JtX2ZyYWN0aW9uL3Iuc2l6ZSsoMS11bmlmb3JtX2ZyYWN0aW9uKSp3ZWlnaHRzL3dlaWdodHMuc3VtKCkKICAgIHJldHVybiBwL3Auc3VtKCkKCgpkZWYgc2FtcGxlX3JhZChjYW5kaWRhdGVzLHJlc2lkdWFsLG5fcG9pbnRzLHJuZyxwb3dlcj0yLix1bmlmb3JtX2ZyYWN0aW9uPS41KToKICAgIHBvaW50cz1ucC5hc2FycmF5KGNhbmRpZGF0ZXMsZHR5cGU9bnAuZmxvYXQ2NCkKICAgIGlmIHBvaW50cy5uZGltIT0yIG9yIGxlbihwb2ludHMpPT0wIG9yIG5vdCBucC5pc2Zpbml0ZShwb2ludHMpLmFsbCgpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ2NhbmRpZGF0ZXMgbXVzdCBiZSBhIGZpbml0ZSBub25lbXB0eSBwb2ludCBtYXRyaXgnKQogICAgaWYgdHlwZShuX3BvaW50cykgaXMgbm90IGludCBvciBuX3BvaW50czwxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ25fcG9pbnRzIG11c3QgYmUgYSBwb3NpdGl2ZSBpbnRlZ2VyJykKICAgIHA9cmVzaWR1YWxfcHJvYmFiaWxpdGllcyhyZXNpZHVhbCxwb3dlcix1bmlmb3JtX2ZyYWN0aW9uKQogICAgaWYgbGVuKHApIT1sZW4ocG9pbnRzKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdyZXNpZHVhbCBhbmQgY2FuZGlkYXRlIGNvdW50cyBkaWZmZXInKQogICAgaW5kaWNlcz1ybmcuY2hvaWNlKGxlbihwb2ludHMpLHNpemU9bl9wb2ludHMscmVwbGFjZT1UcnVlLHA9cCkKICAgIHJldHVybiB7J3BvaW50cyc6cG9pbnRzW2luZGljZXNdLmNvcHkoKSwnaW5kaWNlcyc6aW5kaWNlcywncHJvYmFiaWxpdGllcyc6cCwKICAgICAgICAgICAgJ2VmZmVjdGl2ZV9jYW5kaWRhdGVzJzpmbG9hdCgxL25wLnNxdWFyZShwKS5zdW0oKSksCiAgICAgICAgICAgICd1bmlxdWVfY291bnQnOmludChucC51bmlxdWUoaW5kaWNlcykuc2l6ZSl9Cgo=', 'heat-seed-901.npz': '...', 'heat-seed-902.npz': '...', 'heat-seed-903.npz': '...', 'heat-seed-904.npz': '...', 'heat-seed-905.npz': '...'}
for k, v in BLOB.items(): (p / k).write_bytes(base64.b64decode(v))
n = torch.cuda.device_count(); names = [torch.cuda.get_device_name(i) for i in range(n)]
print(json.dumps({"gpus": names, "hashes": {'heat_rad_runner.py': '73f0e2781c7c7832dff70fd4dd9b8f251f0c4fc18334616eeabfbd3be58c77bd', 'af_sciml_model.py': '934687a265c2b2cae0c375cc30365e2d52e0b1d3edfff7cd977db0b59e4d7767', 'rad.py': '418d7dca9ce3cd9c7a37eba3d7b497c608ded6e0ca226cd06d6549a505afa241', 'heat-seed-901.npz': 'c0901df4a0aaad05656bfa779297c77774590b393e06156b6cdd300a0d425213', 'heat-seed-902.npz': '357844a4bee6599a327f0c44eef116b30be51070342a9286f29f6c5a63e8a314', 'heat-seed-903.npz': '3eb8aed4319f1c96f91049c3429020bc5f25c820f7440f4a94589a7f5296c5da', 'heat-seed-904.npz': 'cec4e6613f3154cdb78b495cb0d0f76caccbb2ef74375a23913f5e0a1a21e291', 'heat-seed-905.npz': '6b9e64f1631befd801be0bdc320c6498396d4d6fb9dbaec8b5f4b07a55d66013'}}), flush=True)
assert n >= 1 and all("T4" in x for x in names)
# 5 RAD jobs first (the pre-registered runs), then 5 control replicates (refresh 0, no L-BFGS) as a parity check of this runner vs EXP-0008
queue = [("rad", s, ["--refresh", "1000", "--lbfgs-seconds", "700"]) for s in range(901, 906)] + \
        [("ctl", s, ["--refresh", "0", "--lbfgs-seconds", "0"]) for s in range(901, 906)]
running, done = {}, []
while queue or running:
    for g in list(running):
        pr, tag, st = running[g]
        if pr.poll() is not None: done.append({"job": tag, "rc": pr.returncode, "wall": time.monotonic() - st}); del running[g]
    if time.monotonic() - T0 > CUT: queue = []
    if time.monotonic() - T0 > CAP:
        for g, (pr, tag, st) in running.items(): pr.kill()
        done.append({"note": "cap reached, killed", "running": [t for _, t, _ in running.values()]}); break
    for g in range(min(n, 2)):
        if g not in running and queue:
            kind, s, extra = queue.pop(0); tag = f"{kind}-{s}"; env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
            cmd = [sys.executable, str(p / "heat_rad_runner.py"), str(s), "--dir", str(p), "--out", str(w / "results" / kind), "--device", "cuda:0", *extra]
            running[g] = (subprocess.Popen(cmd, stdout=open(w / f"{tag}.log", "w"), stderr=subprocess.STDOUT, env=env), tag, time.monotonic())
    time.sleep(3)
summ = {"done": done, "wall": time.monotonic() - T0, "cap": CAP, "scientific_accepted": False, "reference_accepted": False}
(w / "session_summary.json").write_text(json.dumps(summ, indent=1)); print(json.dumps(summ))
